# Training Wide ResNets on Tiny ImageNet

## Complete Beginner's Guide

This notebook trains **wide ResNet classifiers** on Tiny ImageNet - a smaller version of ImageNet with 200 classes and 64x64 images. We progressively build more sophisticated architectures, culminating in a **wide model** with many channels per layer.

### What You'll Learn

1. **Tiny ImageNet Dataset**: Loading and processing 200-class 64x64 images
2. **Custom Dataset Classes**: Building PyTorch datasets from scratch
3. **Data Augmentation**: Padding, cropping, flipping, random erase, and TrivialAugmentWide
4. **ResNet Architecture**: Building residual networks with configurable depth
5. **Pre-activation ResNets**: BN-ReLU-Conv ordering for better gradient flow
6. **Wide Networks**: Increasing channels instead of depth
7. **Training Techniques**: OneCycleLR, mixed precision, AdamW

### Prerequisites
- Basic Python and PyTorch knowledge
- Understanding of CNNs and residual connections

### Wide vs Deep Networks

| Aspect | Deep Network | Wide Network |
|--------|--------------|---------------|
| Architecture | Many layers, fewer channels | Fewer layers, more channels |
| Parameters | Spread across depth | Concentrated in width |
| Training | Harder (vanishing gradients) | Easier to optimize |
| Parallelism | Sequential computation | Better GPU utilization |
| This notebook | 32-64-128-256-512-1024 | 32-64-128-512-1024-1536 |

## Google Colab Setup

Run the cells below **once** at the start of each Colab session. They mount Google Drive, set the working directory, install required packages, and clone the `miniai` library.

**GPU note:** Wider models need more GPU memory. Use a T4 or higher.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir('/content/drive/MyDrive/Fast.AI_Colab')
print(os.getcwd())

In [ ]:
!pip install -q fastcore fastai diffusers datasets torcheval accelerate wandb
if not os.path.exists('course22p2'):
    !git clone https://github.com/fastai/course22p2.git
import sys
sys.path.insert(0, os.path.join(os.getcwd(), 'course22p2'))
try:
    import miniai
    print(f'miniai loaded from: {miniai.__file__}')
except ImportError:
    print('ERROR: miniai not found.')

---

*The cells above are Colab-specific setup. The content below is the same as the source notebook (`24_imgnet_tiny-wide_explained.ipynb`), unchanged.*

---

---
## Part 1: Setup and Imports

In [ ]:
# Select which GPU to use (useful when multiple GPUs are available)
# This MUST be set BEFORE importing torch to take effect
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '2'

In [ ]:
# Core libraries
import shutil                              # File operations (unpacking archives)
import timm                                # PyTorch Image Models library
import os                                  # Operating system utilities
import torch                               # PyTorch deep learning framework
import random                              # Random number generation
import datasets                            # HuggingFace datasets
import math                                # Mathematical functions

# Data manipulation and visualization
import fastcore.all as fc                  # fastcore utilities
import numpy as np                         # Numerical computing
import matplotlib as mpl                   # Plotting configuration
import matplotlib.pyplot as plt            # Plotting

# Deep learning specific
import k_diffusion as K                    # Diffusion utilities (for some helpers)
import torchvision.transforms as T         # Image transformations
import torchvision.transforms.functional as TF  # Functional transforms
import torch.nn.functional as F            # Neural network functions

# PyTorch utilities
from torch.utils.data import DataLoader, default_collate
from pathlib import Path                   # Object-oriented file paths
from torch.nn import init                  # Weight initialization
from fastcore.foundation import L          # List with extra methods
from torch import nn, tensor               # Neural network modules
from operator import itemgetter            # Efficient item extraction
from torcheval.metrics import MulticlassAccuracy  # Accuracy metric
from functools import partial              # Partial function application
from torch.optim import lr_scheduler       # Learning rate schedulers
from torch import optim                    # Optimizers
from torchvision.io import read_image, ImageReadMode  # Fast image loading
from glob import glob                      # File pattern matching

# miniai modules (custom training framework)
from miniai.datasets import *              # Dataset utilities
from miniai.conv import *                  # Convolution helpers
from miniai.learner import *               # Training loop
from miniai.activations import *           # Activation functions
from miniai.init import *                  # Weight initialization
from miniai.sgd import *                   # Optimizers
from miniai.resnet import *                # ResNet blocks
from miniai.augment import *               # Data augmentation
from miniai.accel import *                 # Acceleration (mixed precision)
from miniai.training import *              # Training utilities

In [ ]:
# Progress bars for training
from fastprogress import progress_bar

In [ ]:
# Configure display and reproducibility
torch.set_printoptions(precision=5, linewidth=140, sci_mode=False)  # Cleaner tensor printing
torch.manual_seed(1)                       # Reproducible random numbers
mpl.rcParams['figure.dpi'] = 70            # Figure resolution

set_seed(42)                               # Set all random seeds for reproducibility

# Limit CPU workers to avoid memory issues
if fc.defaults.cpus > 8: 
    fc.defaults.cpus = 8

---
## Part 2: Loading Tiny ImageNet Dataset

Tiny ImageNet is a subset of ImageNet with:
- **200 classes** (vs 1000 in full ImageNet)
- **64x64 images** (vs 224x224+)
- **100,000 training images** (500 per class)
- **10,000 validation images** (50 per class)

This makes it ideal for experimentation while still being challenging.

In [ ]:
# Create data directory
path_data = Path('data')
path_data.mkdir(exist_ok=True)

# Path to Tiny ImageNet
path = path_data / 'tiny-imagenet-200'

# Download URL (Stanford CS231n course hosts this dataset)
url = 'http://cs231n.stanford.edu/tiny-imagenet-200.zip'

# Download and extract if not already present
if not path.exists():
    path_zip = fc.urlsave(url, path_data)   # Download
    shutil.unpack_archive('data/tiny-imagenet-200.zip', 'data')  # Extract

# Batch size - 512 works well for 64x64 images on modern GPUs
bs = 512

### Dataset Structure

```
tiny-imagenet-200/
├── train/
│   ├── n01443537/              # Class folder (WordNet ID)
│   │   ├── images/
│   │   │   ├── n01443537_0.JPEG
│   │   │   └── ... (500 images per class)
│   │   └── n01443537_boxes.txt
│   └── ... (200 class folders)
├── val/
│   ├── images/
│   │   ├── val_0.JPEG
│   │   └── ... (10,000 images, all in one folder)
│   └── val_annotations.txt     # Maps filenames to classes
├── wnids.txt                   # List of 200 WordNet IDs used
└── words.txt                   # WordNet ID to human-readable name
```

### Training Dataset Class

PyTorch datasets need two methods:
- `__len__`: Return total number of samples
- `__getitem__`: Return a single sample by index

In [ ]:
class TinyDS:
    """
    Dataset for Tiny ImageNet training data.
    
    Training images are organized by class folder:
    train/n01443537/images/n01443537_0.JPEG
                    ↑
            class ID is parent.parent of image file
    """
    def __init__(self, path):
        self.path = Path(path)
        # glob finds all files matching a pattern
        # '**/*.JPEG' means: any subdirectory (**), any filename (*), with .JPEG extension
        # recursive=True allows ** to match nested directories
        self.files = glob(str(path / '**/*.JPEG'), recursive=True)
    
    def __len__(self): 
        return len(self.files)
    
    def __getitem__(self, i): 
        filepath = self.files[i]
        # Extract class from path structure:
        # .../train/n01443537/images/file.JPEG
        #          ↑ parent.parent of file
        class_id = Path(filepath).parent.parent.name
        return filepath, class_id

# Create training dataset
tds = TinyDS(path / 'train')

### Validation Dataset

The validation set has a different structure - all images are in one folder, with class labels stored in `val_annotations.txt`.

In [ ]:
# Load validation annotations
# File format: filename<TAB>class_id<TAB>bbox_x<TAB>bbox_y<TAB>bbox_w<TAB>bbox_h
path_anno = path / 'val' / 'val_annotations.txt'

# Create dictionary: filename -> class_id
# Split each line by tab, take first 2 elements (filename, class_id)
anno = dict(
    o.split('\t')[:2] 
    for o in path_anno.read_text().splitlines()
)

# Understanding Tiny ImageNet Validation Annotations Parsing

## Context from the Notebook

This code works with **Tiny ImageNet** dataset, which has:
- 200 classes
- 64×64 pixel images
- Training and validation sets
- The validation set has annotations in a text file

## Breaking Down the Code

### Line 1: Setting the path
```python
path_anno = path / 'val' / 'val_annotations.txt'
```
This points to the annotation file. The file contains bounding box information and class labels.

### Line 2-3: The Dictionary Comprehension

```python
anno = dict(
    o.split('\t')[:2] 
    for o in path_anno.read_text().splitlines()
)
```

## Step-by-Step Execution

### Step 1: Read the entire file as one string
`path_anno.read_text()` reads:
```
"val_0.JPEG\tn03444034\t0\t32\t44\t62\nval_1.JPEG\tn04067472\t52\t55\t57\t59\n..."
```

### Step 2: Split into lines
`.splitlines()` creates a list:
```python
[
    "val_0.JPEG\tn03444034\t0\t32\t44\t62",
    "val_1.JPEG\tn04067472\t52\t55\t57\t59",
    "val_2.JPEG\tn04070727\t4\t0\t60\t55",
    ...
]
```

### Step 3: Split each line by tab
For `"val_0.JPEG\tn03444034\t0\t32\t44\t62"`:
```python
o.split('\t') → ['val_0.JPEG', 'n03444034', '0', '32', '44', '62']
#                     ↑            ↑          ↑    ↑    ↑    ↑
#                 filename    class_id     bbox coordinates
```

### Step 4: Take only first 2 elements
```python
[:2] → ['val_0.JPEG', 'n03444034']
```

### Step 5: Convert to dictionary
`dict()` converts these pairs into:
```python
anno = {
    'val_0.JPEG': 'n03444034',
    'val_1.JPEG': 'n04067472',
    'val_2.JPEG': 'n04070727',
    'val_3.JPEG': 'n02808440',
    'val_4.JPEG': 'n02808440',
    'val_5.JPEG': 'n04399382',
    ...
}
```

## What This Accomplishes

The code creates a **lookup dictionary** that maps:
- **Key**: Image filename (e.g., `'val_0.JPEG'`)
- **Value**: Class ID (e.g., `'n03444034'`)

### Why ignore the bounding box coordinates?

This code is for **image classification** (predicting the class), not object detection. The bounding boxes are provided in the dataset but aren't needed for this task. The code only extracts the class label for each validation image.

## Practical Example

Later in your code, you can use this dictionary like:

```python
# Get the class for a specific image
class_id = anno['val_0.JPEG']  # Returns 'n03444034'
class_id = anno['val_5.JPEG']  # Returns 'n04399382'
```

This is much more efficient than reading the text file every time you need to look up an image's class!

## The Column Meanings

From the validation annotations file:
- **Column 0**: Filename (`val_0.JPEG`)
- **Column 1**: Class ID (`n03444034`) ← **This is what we keep**
- **Columns 2-5**: Bounding box (x, y, width, height) ← **These are ignored**

## Example Data from Your Screenshot

```
val_0.JPEG    n03444034    0     32    44    62
val_1.JPEG    n04067472    52    55    57    59
val_2.JPEG    n04070727    4     0     60    55
val_3.JPEG    n02808440    3     3     63    63
val_4.JPEG    n02808440    9     27    63    48
val_5.JPEG    n04399382    7     0     59    63
```

The compact dictionary comprehension efficiently extracts just what's needed: a filename-to-class mapping for validation.

## Complete Code with Comments

```python
# Load validation annotations
# File format: filename<TAB>class_id<TAB>bbox_x<TAB>bbox_y<TAB>bbox_w<TAB>bbox_h
path_anno = path / 'val' / 'val_annotations.txt'

# Create dictionary: filename -> class_id
# Split each line by tab, take first 2 elements (filename, class_id)
anno = dict(
    o.split('\t')[:2] 
    for o in path_anno.read_text().splitlines()
)

# Now you can look up any image's class:
# anno['val_0.JPEG'] → 'n03444034'
```

## Why This Pattern is Useful

1. **Memory efficient**: Stores only what's needed (filename and class)
2. **Fast lookups**: Dictionary access is O(1)
3. **Clean code**: One-liner using dictionary comprehension
4. **No external dependencies**: Uses only Python built-ins

In [ ]:
class TinyValDS(TinyDS):
    """
    Dataset for Tiny ImageNet validation data.
    
    Inherits from TinyDS but overrides __getitem__ to look up
    class from annotations file instead of folder structure.
    """
    def __getitem__(self, i): 
        filepath = self.files[i]
        # Get filename (e.g., 'val_4619.JPEG')
        filename = os.path.basename(filepath)
        # Look up class in annotations dictionary
        class_id = anno[filename]
        return filepath, class_id

In [ ]:
# Create validation dataset
vds = TinyValDS(path / 'val')

---
## Part 3: Transform Dataset Wrapper

A flexible wrapper that applies transforms to both inputs (images) and targets (labels).

In [ ]:
class TfmDS:
    """
    Transform wrapper for datasets.
    
    Wraps any dataset and applies specified transforms to x and y.
    Uses fc.noop (a do-nothing function) as default when no transform needed.
    
    Args:
        ds: Base dataset
        tfmx: Transform for x (images)
        tfmy: Transform for y (labels)
    """
    def __init__(self, ds, tfmx=fc.noop, tfmy=fc.noop): 
        self.ds = ds
        self.tfmx = tfmx
        self.tfmy = tfmy
    
    def __len__(self): 
        return len(self.ds)
    
    def __getitem__(self, i):
        x, y = self.ds[i]
        return self.tfmx(x), self.tfmy(y)

# Understanding TfmDS Transform Wrapper Class

## The Code

```python
class TfmDS:
    """
    Transform wrapper for datasets.
    
    Wraps any dataset and applies specified transforms to x and y.
    Uses fc.noop (a do-nothing function) as default when no transform needed.
    
    Args:
        ds: Base dataset
        tfmx: Transform for x (images)
        tfmy: Transform for y (labels)
    """
    def __init__(self, ds, tfmx=fc.noop, tfmy=fc.noop): 
        self.ds = ds
        self.tfmx = tfmx
        self.tfmy = tfmy
    
    def __len__(self): 
        return len(self.ds)
    
    def __getitem__(self, i):
        x, y = self.ds[i]
        return self.tfmx(x), self.tfmy(y)
```

## Key Questions Answered

### Are `tfmx` and `tfmy` Functions?

**Yes!** They are **functions** (or more precisely, "callables" - anything that can be called with parentheses).

Look at how they're used in `__getitem__`:
```python
return self.tfmx(x), self.tfmy(y)
#            ↑ called with parentheses - it's a function!
```

### What is `fc.noop`?

`fc.noop` is a **do-nothing function** from the fastcore library. It literally just returns whatever you pass to it:

```python
# Conceptually, fc.noop is like:
def noop(x):
    return x

# So when you call it:
fc.noop(5)      # Returns: 5
fc.noop("hi")   # Returns: "hi"
fc.noop(image)  # Returns: image (unchanged)
```

### When No Transform is Provided

If you create a `TfmDS` without specifying transforms:

```python
transformed_ds = TfmDS(base_ds)  # tfmx and tfmy default to fc.noop
```

Then `__getitem__` does this:

```python
def __getitem__(self, i):
    x, y = self.ds[i]
    return self.tfmx(x), self.tfmy(y)  
    # ↓ becomes ↓
    return fc.noop(x), fc.noop(y)
    # ↓ which is equivalent to ↓
    return x, y  # Just returns them unchanged!
```

**Answer: Yes, when tfmx and tfmy don't do anything (are `fc.noop`), `__getitem__` just returns x and y unchanged.**

## Practical Examples

### Example 1: No transforms (identity)
```python
ds = TfmDS(base_ds)  # Uses fc.noop for both
x, y = ds[0]  # Returns image and label unchanged
```

This is equivalent to:
```python
x, y = base_ds[0]  # Direct access to base dataset
```

### Example 2: Transform only images
```python
def normalize(img):
    return (img - img.mean()) / img.std()

ds = TfmDS(base_ds, tfmx=normalize)  # tfmy still uses fc.noop
x, y = ds[0]  
# x is normalized
# y is unchanged
```

### Example 3: Transform only labels
```python
def to_tensor(label):
    return torch.tensor(label)

ds = TfmDS(base_ds, tfmy=to_tensor)  # tfmx still uses fc.noop
x, y = ds[0]
# x is unchanged
# y is converted to tensor
```

### Example 4: Transform both
```python
def resize_img(img):
    return resize(img, (224, 224))

def to_tensor(label):
    return torch.tensor(label)

ds = TfmDS(base_ds, tfmx=resize_img, tfmy=to_tensor)
x, y = ds[0]
# x is resized to 224x224
# y is converted to tensor
```

## Visual Flow Diagram

```
Without transforms (fc.noop):
base_ds[0] → (x, y) → fc.noop(x), fc.noop(y) → (x, y)  [unchanged]

With image transform:
base_ds[0] → (x, y) → normalize(x), fc.noop(y) → (x_normalized, y)

With both transforms:
base_ds[0] → (x, y) → resize(x), to_tensor(y) → (x_resized, y_tensor)
```

## Why This Design Pattern?

This is a **flexible wrapper pattern** with several advantages:

### 1. Reusability
You can reuse the same dataset with different transforms:
```python
train_ds = TfmDS(base_ds, tfmx=train_augmentation)
valid_ds = TfmDS(base_ds, tfmx=valid_augmentation)  # Different transform!
```

### 2. Safe Defaults
Default behavior is "do nothing" (`fc.noop`), which is safe:
```python
ds = TfmDS(base_ds)  # Works fine, doesn't break
```

### 3. Selective Transformation
You can selectively transform just x, just y, or both:
```python
# Only transform images
ds1 = TfmDS(base_ds, tfmx=resize)

# Only transform labels  
ds2 = TfmDS(base_ds, tfmy=one_hot_encode)

# Transform both
ds3 = TfmDS(base_ds, tfmx=resize, tfmy=one_hot_encode)
```

### 4. Separation of Concerns
The base dataset doesn't need to know about transforms:
- Base dataset: Handles data loading
- TfmDS wrapper: Handles transformations
- Clean, modular code

## Key Insight

**`fc.noop` makes the default behavior explicit and safe.**

If you don't specify a transform:
- Nothing breaks ✓
- Data passes through unchanged ✓
- Code remains simple and readable ✓

## Common Use Cases

### Training vs Validation
```python
# Training: aggressive augmentation
train_tfm = compose([
    random_crop,
    random_flip,
    color_jitter,
    normalize
])

train_ds = TfmDS(base_ds, tfmx=train_tfm)

# Validation: minimal processing
valid_tfm = compose([
    center_crop,
    normalize
])

valid_ds = TfmDS(base_ds, tfmx=valid_tfm)
```

### Progressive Training
```python
# Stage 1: Small images
ds_stage1 = TfmDS(base_ds, tfmx=resize_64)

# Stage 2: Larger images
ds_stage2 = TfmDS(base_ds, tfmx=resize_128)

# Stage 3: Full resolution
ds_stage3 = TfmDS(base_ds, tfmx=resize_224)
```

## How Python Calls These Functions

When you access an item:
```python
ds = TfmDS(base_ds, tfmx=my_transform)
x, y = ds[0]  # What happens?
```

Python does:
1. Calls `ds.__getitem__(0)`
2. Inside `__getitem__`: `x, y = self.ds[0]` (gets from base dataset)
3. Returns `self.tfmx(x), self.tfmy(y)` (applies transforms)
4. Since `tfmx=my_transform` and `tfmy=fc.noop`:
   - `self.tfmx(x)` → `my_transform(x)` → transformed image
   - `self.tfmy(y)` → `fc.noop(y)` → unchanged label

## Summary

- **`tfmx` and `tfmy` are functions**: They get called with `()` on the data
- **`fc.noop` is a pass-through function**: Returns input unchanged
- **Default behavior**: If no transforms specified, data passes through unchanged
- **Flexible design**: Can transform x only, y only, or both
- **Clean pattern**: Separates data loading from data transformation

This is a fundamental pattern in PyTorch/fast.ai for building flexible data pipelines!

### Label Encoding

Convert WordNet IDs (strings like 'n01443537') to integer indices (0-199).

In [ ]:
# Load list of 200 WordNet IDs
id2str = (path / 'wnids.txt').read_text().splitlines()

# Create reverse mapping: string -> integer index
str2id = {v: k for k, v in enumerate(id2str)}

### Image Normalization

Neural networks train better when inputs have **zero mean** and **unit variance**. We normalize using pre-computed dataset statistics.

**Normalization formula:**
$$x_{norm} = \frac{x - \mu}{\sigma}$$

Where $\mu$ is the mean and $\sigma$ is the standard deviation.

In [ ]:
# Pre-computed mean and std for Tiny ImageNet (per RGB channel)
xmean = tensor([0.47565, 0.40303, 0.31555])  # RGB means
xstd = tensor([0.28858, 0.24402, 0.26615])   # RGB standard deviations

In [ ]:
def tfmx(x):
    """
    Transform for images:
    1. Load image from file path
    2. Convert to RGB tensor [0, 1]
    3. Normalize by mean and std
    
    Args:
        x: File path string
    
    Returns:
        Normalized tensor of shape (3, 64, 64)
    """
    # read_image returns tensor with values 0-255
    img = read_image(x, mode=ImageReadMode.RGB) / 255
    # Normalize: (x - mean) / std
    # [:,None,None] reshapes (3,) to (3,1,1) for broadcasting with (3,64,64) image
    return (img - xmean[:, None, None]) / xstd[:, None, None]

def tfmy(y): 
    """Convert WordNet ID string to integer tensor."""
    return tensor(str2id[y])

# Create transformed datasets
tfm_tds = TfmDS(tds, tfmx, tfmy)
tfm_vds = TfmDS(vds, tfmx, tfmy)

In [ ]:
def denorm(x): 
    """
    Reverse normalization for visualization.
    
    Reverses: x_orig = x_norm * std + mean
    Clips to [0, 1] for display.
    """
    return (x * xstd[:, None, None] + xmean[:, None, None]).clip(0, 1)

### Human-Readable Class Names

The `words.txt` file maps WordNet IDs to descriptive names.

In [ ]:
# Load all synset (WordNet) definitions
all_synsets = [o.split('\t') for o in (path / 'words.txt').read_text().splitlines()]

# Create dictionary: WordNet ID -> human name (first comma-separated part only)
synsets = {
    k: v.split(',', maxsplit=1)[0]  # Take first name before comma
    for k, v in all_synsets 
    if k in id2str  # Only include our 200 classes
}

# Understanding `maxsplit=1` in Python's `split()` Method

## The Code in Question

```python
synsets = {
    k: v.split(',', maxsplit=1)[0]  # Take first name before comma
    for k, v in all_synsets 
    if k in id2str  # Only include our 200 classes
}
```

## What `maxsplit=1` Does

The `maxsplit=1` parameter in `split()` tells Python to split the string **at most once**, resulting in **at most 2 parts**.

---

## Example

Let's say you have a WordNet entry like:

```python
v = "golden retriever, yellow retriever, hunting dog"
```

### Without `maxsplit`:

```python
v.split(',')
# Result: ['golden retriever', ' yellow retriever', ' hunting dog']
# (3 parts - splits at every comma)
```

### With `maxsplit=1`:

```python
v.split(',', maxsplit=1)
# Result: ['golden retriever', ' yellow retriever, hunting dog']
# (2 parts - splits only at the FIRST comma)
```

---

## What Happens in the Code

```python
v.split(',', maxsplit=1)[0]
```

This takes only the **first element** `[0]` after the split, which gives you `'golden retriever'` — the primary/preferred name for the class.

---

## Why Do It This Way?

The WordNet definitions often contain multiple synonyms separated by commas. The code only wants the **first, most common name** for display purposes. 

Using `maxsplit=1` is slightly more efficient than splitting the entire string when you only need the first part anyway — Python doesn't need to process the rest of the string looking for more commas.

In [ ]:
# Create DataLoaders (combines train and validation into one object)
dls = DataLoaders(*get_dls(tfm_tds, tfm_vds, bs=bs, num_workers=8))

---
## Part 4: Data Augmentation

Data augmentation creates variations of training images to:
1. **Increase effective dataset size**
2. **Prevent overfitting**
3. **Teach invariance** (model learns to recognize objects despite transformations)

In [ ]:
def tfm_batch(b, tfm_x=fc.noop, tfm_y=fc.noop): 
    """
    Apply transforms to a batch.
    
    Args:
        b: Tuple of (inputs, targets)
        tfm_x: Transform for inputs
        tfm_y: Transform for targets
    
    Returns:
        Transformed (inputs, targets)
    """
    return tfm_x(b[0]), tfm_y(b[1])

In [ ]:
# Define augmentation transforms as a sequential pipeline
tfms = nn.Sequential(
    T.Pad(4),              # Add 4 pixels padding on each side: 64 -> 72
    T.RandomCrop(64),      # Random crop back to 64x64 (introduces translation)
    T.RandomHorizontalFlip(),  # 50% chance of horizontal flip
    RandErase()            # Randomly erase a rectangle (cutout regularization)
)

# Create callback that applies augmentation to training batches only
# on_val=False means validation data is NOT augmented
augcb = BatchTransformCB(partial(tfm_batch, tfm_x=tfms), on_val=False)

**Augmentation Pipeline Visualization:**

```
Original 64x64
      ↓
Pad(4) → 72x72 (4 pixels added to each side)
      ↓
RandomCrop(64) → 64x64 (random position within 72x72)
      ↓                  (this creates random translations up to ±4 pixels)
RandomHorizontalFlip → 50% flipped
      ↓
RandErase → Random rectangle set to mean color
```

### Activation Function and Weight Initialization

In [ ]:
# GeneralRelu: Leaky ReLU with subtraction to center activations
# f(x) = max(0.1*x, x) - 0.4
# leak=0.1: allows small negative gradients (prevents "dead" neurons)
# sub=0.4: shifts activations to be centered around zero
act_gr = partial(GeneralRelu, leak=0.1, sub=0.4)

# Kaiming initialization adjusted for leaky ReLU
iw = partial(init_weights, leaky=0.1)

**Why GeneralRelu?**

Standard ReLU: $f(x) = \max(0, x)$
- Problem: Neurons that output 0 have zero gradient ("dead neurons")
- Problem: Outputs are always positive (not centered)

GeneralRelu: $f(x) = \max(0.1x, x) - 0.4$
- Leaky slope (0.1) allows gradients to flow even for negative inputs
- Subtraction (0.4) centers the distribution around zero

---
## Part 5: Initial ResNet Model

Our first model uses single ResBlocks at each stage with increasing channel counts.

In [ ]:
# Number of filters (channels) at each stage
# Progressive widening: 32 -> 64 -> 128 -> 256 -> 512 -> 1024
nfs = (32, 64, 128, 256, 512, 1024)

def get_dropmodel(act=act_gr, nfs=nfs, norm=nn.BatchNorm2d, drop=0.1):
    """
    Create a ResNet classifier with single blocks per stage.
    
    Architecture:
    - Initial 5x5 conv (captures larger patterns at input)
    - 5 ResBlocks with stride=2 (each halves spatial dimensions)
    - Global average pooling
    - Dropout
    - Linear to 200 classes
    
    Args:
        act: Activation function
        nfs: Tuple of filter counts per stage
        norm: Normalization layer type
        drop: Dropout probability
    """
    layers = [nn.Conv2d(3, nfs[0], 5, padding=2)]  # 3->32, keep spatial size
    
    # ResBlocks: each doubles channels and halves spatial dimensions
    layers += [ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2)
               for i in range(len(nfs)-1)]
    
    # Classification head
    layers += [
        nn.AdaptiveAvgPool2d(1),  # Global average pooling: (N,C,H,W) -> (N,C,1,1)
        nn.Flatten(),              # (N,C,1,1) -> (N,C)
        nn.Dropout(drop),          # Regularization
    ]
    
    # Final linear layer (bias=False because BatchNorm handles bias)
    layers += [nn.Linear(nfs[-1], 200, bias=False), nn.BatchNorm1d(200)]
    
    return nn.Sequential(*layers).apply(iw)

**Network spatial dimensions:**

```
Input:      3 x 64 x 64
Conv 5x5:  32 x 64 x 64
ResBlock:  64 x 32 x 32  (stride=2)
ResBlock: 128 x 16 x 16  (stride=2)
ResBlock: 256 x  8 x  8  (stride=2)
ResBlock: 512 x  4 x  4  (stride=2)
ResBlock:1024 x  2 x  2  (stride=2)
AvgPool: 1024 x  1 x  1
Linear:  200 (classes)
```

---
## Part 6: Deeper Model with Multiple Blocks per Stage

To increase model capacity, we can use **multiple ResBlocks** at each stage.

In [ ]:
def res_blocks(n_bk, ni, nf, stride=1, ks=3, act=act_gr, norm=None):
    """
    Create a sequence of ResNet blocks.
    
    Key insight: Only the LAST block has stride>1 (for downsampling).
    Earlier blocks preserve spatial dimensions.
    
    Args:
        n_bk: Number of blocks
        ni: Input channels
        nf: Output channels
        stride: Stride for the LAST block (others have stride=1)
        ks: Kernel size
        act: Activation function
        norm: Normalization layer
    
    Returns:
        nn.Sequential of ResBlocks
    
    Example with n_bk=3, ni=64, nf=128, stride=2:
        Block 0: 64->128, stride=1  (channel change, keep size)
        Block 1: 128->128, stride=1 (same channels, same size)
        Block 2: 128->128, stride=2 (same channels, halve size)
    """
    return nn.Sequential(*[
        ResBlock(
            ni if i == 0 else nf,           # First block: ni->nf, others: nf->nf
            nf,
            stride=stride if i == n_bk - 1 else 1,  # Only last block downsamples
            ks=ks,
            act=act,
            norm=norm
        )
        for i in range(n_bk)
    ])

In [ ]:
# Number of blocks at each stage
# More blocks at earlier stages (more spatial resolution to process)
nbks = (3, 2, 2, 1, 1)

def get_dropmodel(act=act_gr, nfs=nfs, nbks=nbks, norm=nn.BatchNorm2d, drop=0.2):
    """
    Deeper ResNet with multiple blocks per stage.
    
    Total blocks: sum(nbks) = 3+2+2+1+1 = 9 ResBlocks
    """
    # Start with a ResBlock instead of plain conv
    layers = [ResBlock(3, nfs[0], ks=5, stride=1, act=act, norm=norm)]
    
    # Multiple ResBlocks at each stage
    layers += [res_blocks(nbks[i], nfs[i], nfs[i+1], act=act, norm=norm, stride=2)
               for i in range(len(nfs)-1)]
    
    # Classification head
    layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(drop)]
    layers += [nn.Linear(nfs[-1], 200, bias=False), nn.BatchNorm1d(200)]
    
    return nn.Sequential(*layers).apply(iw)

---
## Part 7: Training Setup

In [ ]:
# AdamW optimizer with custom epsilon for numerical stability
# eps=1e-5 (vs default 1e-8) prevents division instabilities with mixed precision
opt_func = partial(optim.AdamW, eps=1e-5)

In [ ]:
# Training callbacks
metrics = MetricsCB(accuracy=MulticlassAccuracy())  # Track accuracy
cbs = [
    DeviceCB(),              # Move data to GPU
    metrics,                 # Track metrics
    ProgressCB(plot=True),   # Show progress bar and plot
    MixedPrecision()         # FP16 training for speed
]

# Training parameters
epochs = 25
lr = 3e-2

# OneCycleLR: Learning rate schedule that warms up then anneals
tmax = epochs * len(dls.train)  # Total training steps
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)

# Additional callbacks: scheduler and augmentation
xtra = [BatchSchedCB(sched), augcb]

# Create learner
learn = Learner(get_dropmodel(), dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=opt_func)

**OneCycleLR Learning Rate Schedule:**

```
LR
↑
|      /\
|     /  \
|    /    \
|   /      \
|  /        \
| /          \
|/____________\___
              → epochs
```

1. **Warmup**: LR increases linearly (helps escape bad local minima)
2. **Annealing**: LR decreases (fine-tunes the solution)

---
## Part 8: Advanced Augmentation with TrivialAugmentWide

**TrivialAugmentWide** randomly applies ONE augmentation from a predefined set, with a random magnitude. It's simple yet very effective.

In [ ]:
# Advanced augmentation pipeline
aug_tfms = nn.Sequential(
    T.Pad(4),                  # Padding for random crop
    T.RandomCrop(64),          # Random translation
    T.RandomHorizontalFlip(),  # Random flip
    T.TrivialAugmentWide()     # Random augmentation from a set
)

# Separate normalization and random erase (applied after TrivialAugment)
norm_tfm = T.Normalize(xmean, xstd)
erase_tfm = RandErase()

**TrivialAugmentWide Operations:**

Randomly selects ONE of these with random magnitude:
- **Geometric**: Rotate, ShearX, ShearY, TranslateX, TranslateY
- **Color**: Brightness, Color, Contrast, Sharpness
- **Other**: Posterize, Solarize, Equalize, AutoContrast, Invert, Identity

The simplicity (one operation per image) is what makes it work well - it avoids the complexity of finding optimal augmentation policies.

In [ ]:
from PIL import Image

def tfmx(x, aug=False):
    """
    Image transform with optional augmentation.
    
    TrivialAugmentWide works on PIL Images, so we:
    1. Load as PIL Image
    2. Apply augmentation (if training)
    3. Convert to tensor
    4. Normalize
    5. Apply random erase (if training)
    
    Args:
        x: File path
        aug: Whether to apply augmentation (True for training)
    """
    # Load as PIL Image (TrivialAugmentWide needs PIL)
    x = Image.open(x).convert('RGB')
    
    # Apply augmentation pipeline (training only)
    if aug: 
        x = aug_tfms(x)
    
    # Convert to tensor (0-1 range)
    x = TF.to_tensor(x)
    
    # Normalize
    x = norm_tfm(x)
    
    # Random erase (training only)
    # Need to add/remove batch dim for RandErase
    if aug: 
        x = erase_tfm(x[None])[0]
    
    return x

In [ ]:
# Create datasets with different augmentation settings
tfm_tds = TfmDS(tds, partial(tfmx, aug=True), tfmy)   # Training: WITH augmentation
tfm_vds = TfmDS(vds, tfmx, tfmy)                       # Validation: NO augmentation

# Recreate DataLoaders with new transforms
dls = DataLoaders(*get_dls(tfm_tds, tfm_vds, bs=bs, num_workers=8))

---
## Part 9: Pre-activation ResNet

In standard ResNets, the order is: **Conv -> BatchNorm -> ReLU**

In pre-activation ResNets, the order is: **BatchNorm -> ReLU -> Conv**

This seemingly small change improves gradient flow in very deep networks.

In [ ]:
def conv(ni, nf, ks=3, stride=1, act=nn.ReLU, norm=None, bias=True):
    """
    Pre-activation convolution: Norm -> Act -> Conv
    
    Traditional order: Conv -> Norm -> Act
    Pre-activation:    Norm -> Act -> Conv
    
    Why pre-activation works better:
    - Gradients flow more directly through identity shortcut
    - BatchNorm acts as a form of regularization at each layer
    - Activations are normalized before each convolution
    """
    layers = []
    if norm: layers.append(norm(ni))   # Normalize input channels
    if act:  layers.append(act())      # Activate
    layers.append(nn.Conv2d(ni, nf, stride=stride, kernel_size=ks, padding=ks//2, bias=bias))
    return nn.Sequential(*layers)

def _conv_block(ni, nf, stride, act=act_gr, norm=None, ks=3):
    """Two pre-activation convolutions."""
    return nn.Sequential(
        conv(ni, nf, stride=1, act=act, norm=norm, ks=ks),      # First conv: change channels
        conv(nf, nf, stride=stride, act=act, norm=norm, ks=ks)  # Second conv: maybe downsample
    )

In [ ]:
class ResBlock(nn.Module):
    """
    Pre-activation ResNet block.
    
    Key differences from standard ResBlock:
    1. Uses pre-activation conv (norm->act->conv)
    2. Uses AvgPool for downsampling instead of strided conv
       (AvgPool preserves more information than strided conv)
    
    Architecture:
        x ─────────────────────────────────┐
        │                                  │ (identity or 1x1 conv)
        │                                  │ (+ AvgPool if stride>1)
        ↓                                  ↓
    BN -> Act -> Conv3x3 -> BN -> Act -> Conv3x3 ──→ (+) ──→ output
    """
    def __init__(self, ni, nf, stride=1, ks=3, act=act_gr, norm=None):
        super().__init__()
        # Main path: two pre-activation convolutions
        self.convs = _conv_block(ni, nf, stride, act=act, ks=ks, norm=norm)
        
        # Shortcut path: identity if channels match, else 1x1 conv
        self.idconv = fc.noop if ni == nf else conv(ni, nf, ks=1, stride=1, act=None, norm=norm)
        
        # Downsampling on shortcut: use AvgPool (not strided conv)
        # ceil_mode=True ensures output size matches main path
        self.pool = fc.noop if stride == 1 else nn.AvgPool2d(2, ceil_mode=True)

    def forward(self, x): 
        # Add main path output to (optionally pooled and projected) shortcut
        return self.convs(x) + self.idconv(self.pool(x))

**Why Average Pooling for Downsampling?**

When we need to downsample the shortcut path (when stride > 1), we have two choices:

1. **Strided 1x1 convolution**: Loses 75% of information (only uses every other pixel)
2. **Average pooling**: Preserves all information by averaging

Average pooling is generally better because:
- No information is discarded
- No additional learnable parameters
- Acts as anti-aliasing

In [ ]:
def get_dropmodel(act=act_gr, nfs=nfs, nbks=nbks, norm=nn.BatchNorm2d, drop=0.2):
    """
    Pre-activation ResNet model.
    
    Note: Since we use pre-activation, we need explicit
    activation and normalization AFTER the last ResBlock
    (before the classification head).
    """
    # Initial conv (not pre-activation - no previous features to normalize)
    layers = [nn.Conv2d(3, nfs[0], 5, padding=2)]
    
    # Pre-activation ResBlocks
    layers += [res_blocks(nbks[i], nfs[i], nfs[i+1], act=act, norm=norm, stride=2)
               for i in range(len(nfs)-1)]
    
    # Final activation and normalization (needed for pre-activation ResNet)
    layers += [act_gr(), norm(nfs[-1])]
    
    # Classification head
    layers += [
        nn.AdaptiveAvgPool2d(1), 
        nn.Flatten(), 
        nn.Dropout(drop)
    ]
    layers += [nn.Linear(nfs[-1], 200, bias=False), nn.BatchNorm1d(200)]
    
    return nn.Sequential(*layers).apply(iw)

---
## Part 10: Training the Wide Model

Now we create a **wide** model - more channels per layer instead of more layers.

In [ ]:
# Training configuration for wide model
epochs = 50
lr = 0.1

# OneCycleLR scheduler
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched)]  # No augcb needed - augmentation is in tfmx

# WIDE MODEL CONFIGURATION
# Note the larger channel counts, especially in later stages:
# Standard: (32, 64, 128, 256, 512, 1024)
# Wide:     (32, 64, 128, 512, 1024, 1536)
#                        ↑    ↑     ↑
#                   2x   2x   1.5x wider

model = get_dropmodel(
    nbks=(1, 2, 8, 2, 2),              # Block counts (many blocks at 128->512 stage)
    nfs=(32, 64, 128, 512, 1024, 1536), # WIDE channel configuration
    drop=0.1                            # Lower dropout for wide model
)

learn = Learner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=opt_func)

**Wide Model Architecture:**

```
Input:       3 x 64 x 64
Conv 5x5:   32 x 64 x 64

Stage 1 (1 block):   32 ->   64 x 32 x 32
Stage 2 (2 blocks):  64 ->  128 x 16 x 16
Stage 3 (8 blocks): 128 ->  512 x  8 x  8  ← Most computation here
Stage 4 (2 blocks): 512 -> 1024 x  4 x  4
Stage 5 (2 blocks):1024 -> 1536 x  2 x  2  ← Very wide!

AvgPool:   1536 x  1 x  1
Linear:    200 (classes)
```

The model is "wide" because later stages have many more channels (1536) than a typical ResNet.

**Why Wide Networks?**

| Deep Network | Wide Network |
|--------------|---------------|
| Many layers, fewer channels | Fewer layers, more channels |
| Gradient flow challenges | Easier optimization |
| Sequential computation | Better parallelism on GPU |
| May need careful initialization | More robust to hyperparameters |

Research (WideResNet paper) showed that widening is often more efficient than deepening.

In [ ]:
# Train the wide model for 50 epochs
learn.fit(epochs)

In [ ]:
# Save the trained model
torch.save(learn.model, 'models/inettiny-wide-50')

---
## Summary

### Key Concepts

**1. Tiny ImageNet Dataset**
- 200 classes, 64x64 images
- 100,000 training / 10,000 validation
- Custom dataset classes: `TinyDS`, `TinyValDS`, `TfmDS`

**2. Data Augmentation Pipeline**
- Basic: Pad + RandomCrop + RandomHorizontalFlip + RandErase
- Advanced: TrivialAugmentWide (random single augmentation)

**3. ResNet Architecture Progression**
- Single blocks per stage → Multiple blocks per stage
- Standard ResBlock → Pre-activation ResBlock (BN-ReLU-Conv)
- Strided conv downsampling → AvgPool downsampling

**4. Wide Model Design**
- Channel progression: 32 → 64 → 128 → 512 → 1024 → 1536
- More channels = more capacity, better GPU utilization
- Block distribution: (1, 2, 8, 2, 2) - most blocks at middle resolution

**5. Training Techniques**
- AdamW optimizer with eps=1e-5
- OneCycleLR learning rate schedule
- Mixed precision (FP16) for speed
- 50 epochs of training

### Architecture Comparison

| Model | Channels | Blocks | Key Feature |
|-------|----------|--------|-------------|
| Basic | 32-64-128-256-512-1024 | (1,1,1,1,1) | Single blocks |
| Deeper | 32-64-128-256-512-1024 | (3,2,2,1,1) | Multiple blocks |
| **Wide** | **32-64-128-512-1024-1536** | **(1,2,8,2,2)** | **Wide channels** |

### fin -